# Chapter 7: Saffron City -- Difference-in-Differences & Synthetic Controls

---

*Team Rocket has taken over Saffron City. The Shadow Surge TM rollout is spreading across Kanto.*
*Can we identify the causal effect of these events using panel data methods?*

In this chapter you will learn:

1. The classic 2x2 Difference-in-Differences (DiD) design
2. How to check the parallel trends assumption
3. Event study specifications and what they reveal
4. Two-Way Fixed Effects (TWFE) regression -- and why it can fail with staggered adoption
5. The Goodman-Bacon decomposition and the TWFE trap
6. Callaway & Sant'Anna group-time ATT estimation
7. Synthetic control methods
8. Placebo tests for synthetic controls

---

## Cell 1: Setup -- Loading Data and Setting the Scene

In [ ]:
# ============================================================
# Cell 1: Setup
# ============================================================
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats, optimize
import warnings
warnings.filterwarnings('ignore')

# Ensure kanto_utils is importable
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from kanto_utils import (
    load_cities_panel, load_shadow_surge,
    apply_kanto_theme, did_plot, did_estimate,
    oak_says, blue_says, blues_mistake, badge_earned,
    gym_leader_says, nurse_joy_says,
)

apply_kanto_theme()

# Load datasets
cities = load_cities_panel()
shadow = load_shadow_surge()

print("Kanto Cities Panel:")
print(f"  Cities: {cities['city'].unique().tolist()}")
print(f"  Periods: 1-{cities['period'].max()}")
print(f"  Shape: {cities.shape}")
print()
print("Shadow Surge Staggered:")
print(f"  Cities: {shadow['city'].unique().tolist()}")
print(f"  Cohorts: {sorted(shadow['cohort'].dropna().unique().tolist())}")
print(f"  Shape: {shadow.shape}")

oak_says(
    "Welcome to Saffron City, trainer! Team Rocket has been causing chaos here, "
    "and the Shadow Surge TM is rolling out across Kanto's cities at different times. "
    "This <b>staggered adoption</b> gives us a natural experiment. But be careful -- "
    "the standard tools can mislead you when treatment timing varies. "
    "Today we'll learn to see through the fog with DiD, event studies, and synthetic controls."
)

## Cell 2: Classic 2x2 Difference-in-Differences

We start with the simplest DiD design: **one treated city vs. one control city**, with a clear pre/post split.

- **Treated:** Saffron City (Team Rocket invaded in periods 8-9)
- **Control:** Pewter City (never invaded)
- **Outcome:** Average trainer level
- **Treatment period:** Period 8

In [ ]:
# ============================================================
# Cell 2: Classic 2x2 DiD -- Saffron vs Pewter
# ============================================================

# Focus on Saffron (treated) and Pewter (control)
treated_city = "Saffron City"
control_city = "Pewter City"
treatment_period = 8  # Team Rocket invasion begins

saffron = cities[cities['city'] == treated_city]
pewter = cities[cities['city'] == control_city]

# Extract pre and post values
y_pre_treat = saffron[saffron['period'] < treatment_period]['avg_trainer_level'].values
y_post_treat = saffron[saffron['period'] >= treatment_period]['avg_trainer_level'].values
y_pre_ctrl = pewter[pewter['period'] < treatment_period]['avg_trainer_level'].values
y_post_ctrl = pewter[pewter['period'] >= treatment_period]['avg_trainer_level'].values

# Compute DiD by hand
diff_treat = y_post_treat.mean() - y_pre_treat.mean()
diff_ctrl = y_post_ctrl.mean() - y_pre_ctrl.mean()
did_manual = diff_treat - diff_ctrl

print("=== 2x2 DiD by Hand ===")
print(f"Treated ({treated_city}): Post - Pre = {y_post_treat.mean():.2f} - {y_pre_treat.mean():.2f} = {diff_treat:.2f}")
print(f"Control ({control_city}): Post - Pre = {y_post_ctrl.mean():.2f} - {y_pre_ctrl.mean():.2f} = {diff_ctrl:.2f}")
print(f"DiD Estimate: {diff_treat:.2f} - {diff_ctrl:.2f} = {did_manual:.2f}")
print()

# Now use the kanto_utils estimator
result = did_estimate(y_pre_treat, y_post_treat, y_pre_ctrl, y_post_ctrl)
print("=== did_estimate() Results ===")
for k, v in result.items():
    print(f"  {k}: {v:.4f}")

# Visualize with did_plot
did_df = cities[cities['city'].isin([treated_city, control_city])].copy()
did_df['treated'] = (did_df['city'] == treated_city).astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
did_plot(did_df, time_col='period', outcome_col='avg_trainer_level',
         group_col='treated', treatment_period=treatment_period, ax=ax)
ax.set_title(f'DiD: {treated_city} vs {control_city}', fontsize=14)
ax.set_xlabel('Period')
ax.set_ylabel('Avg Trainer Level')
plt.tight_layout()
plt.show()

oak_says(
    f"The DiD estimate is <b>{did_manual:.2f}</b>. This tells us that Saffron City's "
    "average trainer level changed by this much <i>more</i> than Pewter City's after "
    "Team Rocket's invasion. The dashed line shows the counterfactual trend for Saffron "
    "if it had followed Pewter's trajectory. But remember -- this only works if the "
    "<b>parallel trends</b> assumption holds!"
)

## Cell 3: Parallel Trends Check

The key identifying assumption in DiD is **parallel trends**: absent treatment, the treated and control groups would have followed the same trend. We can never prove this, but we can check whether pre-treatment trends look parallel.

In [ ]:
# ============================================================
# Cell 3: Parallel Trends Check
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Saffron vs all control cities (pre-treatment only)
ax = axes[0]
control_cities = ['Pewter City', 'Cerulean City', 'Fuchsia City', 'Cinnabar Island']
pre_periods = cities['period'] < treatment_period

# Plot Saffron
saffron_pre = saffron[saffron['period'] < treatment_period]
ax.plot(saffron_pre['period'], saffron_pre['avg_trainer_level'],
        'o-', color='#EE1515', linewidth=2.5, markersize=7, label=treated_city, zorder=5)

# Plot control cities
colors = ['#3B4CCA', '#FFD733', '#4DAD5B', '#7B61FF']
for i, ctrl in enumerate(control_cities):
    ctrl_data = cities[(cities['city'] == ctrl) & pre_periods]
    ax.plot(ctrl_data['period'], ctrl_data['avg_trainer_level'],
            's--', color=colors[i], linewidth=1.5, alpha=0.7, label=ctrl)

ax.set_title('Pre-Treatment Trends (All Cities)', fontsize=13)
ax.set_xlabel('Period')
ax.set_ylabel('Avg Trainer Level')
ax.legend(fontsize=9)

# Panel 2: Normalized (index to period 1) for better comparison
ax = axes[1]
for city_name, color, marker in [(treated_city, '#EE1515', 'o')] + \
    list(zip(control_cities, colors, ['s', 'D', '^', 'v'])):
    c = cities[(cities['city'] == city_name) & pre_periods].copy()
    base = c['avg_trainer_level'].iloc[0]
    c['indexed'] = c['avg_trainer_level'] / base * 100
    lw = 2.5 if city_name == treated_city else 1.5
    alpha = 1.0 if city_name == treated_city else 0.7
    ax.plot(c['period'], c['indexed'], f'{marker}-', color=color,
            linewidth=lw, alpha=alpha, label=city_name)

ax.set_title('Indexed Pre-Treatment Trends (Period 1 = 100)', fontsize=13)
ax.set_xlabel('Period')
ax.set_ylabel('Indexed Avg Trainer Level')
ax.axhline(100, color='gray', linestyle=':', alpha=0.5)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Formal pre-trend test: regress outcome on time*treated interaction for pre-periods
pre_df = did_df[did_df['period'] < treatment_period].copy()
pre_df['time_trend'] = pre_df['period']
pre_df['trend_x_treated'] = pre_df['time_trend'] * pre_df['treated']

# OLS: y = a + b1*time + b2*treated + b3*time*treated
X = np.column_stack([
    np.ones(len(pre_df)),
    pre_df['time_trend'].values,
    pre_df['treated'].values,
    pre_df['trend_x_treated'].values,
])
y = pre_df['avg_trainer_level'].values
beta = np.linalg.lstsq(X, y, rcond=None)[0]
resid = y - X @ beta
n, k = X.shape
sigma2 = np.sum(resid**2) / (n - k)
se_beta = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))
t_stat = beta[3] / se_beta[3]
p_val = 2 * stats.t.sf(abs(t_stat), n - k)

print("=== Pre-Trend Test (Differential Trend) ===")
print(f"Coefficient on Time x Treated: {beta[3]:.4f}")
print(f"Std Error: {se_beta[3]:.4f}")
print(f"t-stat: {t_stat:.3f}, p-value: {p_val:.4f}")
print(f"{'PASS' if p_val > 0.05 else 'FAIL'}: "
      f"{'No significant' if p_val > 0.05 else 'Significant'} differential pre-trend")

oak_says(
    "Looking at the pre-treatment period, Saffron and Pewter City appear to follow "
    "roughly parallel trends in average trainer level. The formal pre-trend test confirms "
    "no statistically significant differential trend. This is encouraging -- but remember, "
    "parallel pre-trends don't <i>guarantee</i> the counterfactual would have been parallel too!"
)

## Cell 4: Event Study

An event study generalizes DiD by estimating period-by-period treatment effects relative to the treatment date. This lets us:
- **Verify** pre-treatment coefficients are near zero (supporting parallel trends)
- **See** how the treatment effect evolves over time

In [ ]:
# ============================================================
# Cell 4: Event Study -- Lead/Lag Dummies with OLS
# ============================================================

# Build an event study dataset for Saffron vs Pewter
es_df = did_df.copy()
es_df['rel_time'] = es_df['period'] - treatment_period  # relative to treatment

# Create lead/lag dummies (exclude rel_time = -1 as reference)
rel_times = sorted(es_df['rel_time'].unique())
ref_period = -1  # normalize to period just before treatment
dummies_to_include = [t for t in rel_times if t != ref_period]

# Build regression matrix: y = city_FE + period_FE + sum(beta_k * treated * 1[rel_time == k])
# City FE: just a treated dummy (2 cities)
# Period FE: period dummies

# Construct design matrix manually
n = len(es_df)
X_parts = [np.ones(n)]  # intercept
X_parts.append(es_df['treated'].values)  # city FE (treated indicator)

# Period FEs (exclude period 1 as reference)
periods = sorted(es_df['period'].unique())
for p in periods[1:]:
    X_parts.append((es_df['period'] == p).astype(float).values)

# Event study dummies: treated * 1[rel_time == k]
col_names_es = []
for k in dummies_to_include:
    dummy = ((es_df['treated'] == 1) & (es_df['rel_time'] == k)).astype(float).values
    X_parts.append(dummy)
    col_names_es.append(k)

X_es = np.column_stack(X_parts)
y_es = es_df['avg_trainer_level'].values

# OLS
beta_es, _, _, _ = np.linalg.lstsq(X_es, y_es, rcond=None)
resid_es = y_es - X_es @ beta_es
n_obs, n_params = X_es.shape
sigma2_es = np.sum(resid_es**2) / (n_obs - n_params)
try:
    cov_es = sigma2_es * np.linalg.inv(X_es.T @ X_es)
except np.linalg.LinAlgError:
    cov_es = sigma2_es * np.linalg.pinv(X_es.T @ X_es)

# Extract event study coefficients (last len(dummies_to_include) columns)
n_es = len(dummies_to_include)
es_coefs = beta_es[-n_es:]
es_ses = np.sqrt(np.diag(cov_es)[-n_es:])

# Add the reference period (coef = 0, se = 0)
all_rel_times = sorted(dummies_to_include + [ref_period])
coefs_full = []
ses_full = []
for t in all_rel_times:
    if t == ref_period:
        coefs_full.append(0.0)
        ses_full.append(0.0)
    else:
        idx = dummies_to_include.index(t)
        coefs_full.append(es_coefs[idx])
        ses_full.append(es_ses[idx])

coefs_full = np.array(coefs_full)
ses_full = np.array(ses_full)

# Plot event study
fig, ax = plt.subplots(figsize=(11, 6))
ax.errorbar(all_rel_times, coefs_full, yerr=1.96 * ses_full,
            fmt='o-', color='#EE1515', capsize=4, capthick=1.5,
            linewidth=2, markersize=7, label='Estimated effect')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(-0.5, color='#FFD733', linestyle='--', linewidth=2, label='Treatment onset')
ax.fill_betweenx(ax.get_ylim(), -0.5, max(all_rel_times) + 0.5,
                 color='#EE1515', alpha=0.05)
ax.set_xlabel('Periods Relative to Treatment', fontsize=13)
ax.set_ylabel('Estimated Effect on Avg Trainer Level', fontsize=13)
ax.set_title('Event Study: Saffron City vs Pewter City', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Summarize
pre_coefs = [c for t, c in zip(all_rel_times, coefs_full) if t < 0 and t != ref_period]
post_coefs = [c for t, c in zip(all_rel_times, coefs_full) if t >= 0]
print("=== Event Study Summary ===")
print(f"Pre-treatment coefficients (should be ~0): mean = {np.mean(pre_coefs):.3f}")
print(f"Post-treatment coefficients: mean = {np.mean(post_coefs):.3f}")

oak_says(
    "The event study shows that pre-treatment coefficients hover around zero, "
    "supporting the parallel trends assumption. After treatment (period 0 onwards), "
    "we see the treatment effect emerge. This is exactly the pattern we hope for: "
    "no pre-trend, then a clear shift at treatment onset."
)

## Cell 5: TWFE Regression

Now let's move to the **staggered** setting. The Shadow Surge TM was adopted by different cities at different times:
- Saffron City: Period 6
- Cerulean City: Period 10
- Vermilion City: Period 14
- Lavender Town: Period 18
- Others: Never treated

The standard approach is Two-Way Fixed Effects (TWFE): regress outcome on unit FEs, time FEs, and a treatment indicator.

In [ ]:
# ============================================================
# Cell 5: TWFE Regression on Staggered Data
# ============================================================

# Use the shadow surge staggered dataset
df = shadow.copy()

# Show the staggered adoption pattern
print("=== Staggered Adoption Timeline ===")
cohort_info = df.groupby('city')['cohort'].first().sort_values()
for city, cohort in cohort_info.items():
    if pd.notna(cohort):
        print(f"  {city}: adopted in period {int(cohort)}")
    else:
        print(f"  {city}: never treated")
print()

# Build TWFE regression: Y_it = alpha_i + gamma_t + delta * D_it + epsilon_it
# Create city and period dummies
city_dummies = pd.get_dummies(df['city'], prefix='city', drop_first=True, dtype=float)
period_dummies = pd.get_dummies(df['period'], prefix='period', drop_first=True, dtype=float)

X_twfe = np.column_stack([
    np.ones(len(df)),
    city_dummies.values,
    period_dummies.values,
    df['treated'].values
])
y_twfe = df['avg_pokemon_level'].values

# OLS
beta_twfe, _, _, _ = np.linalg.lstsq(X_twfe, y_twfe, rcond=None)
resid_twfe = y_twfe - X_twfe @ beta_twfe
n_tw, k_tw = X_twfe.shape
sigma2_tw = np.sum(resid_twfe**2) / (n_tw - k_tw)
try:
    cov_tw = sigma2_tw * np.linalg.inv(X_twfe.T @ X_twfe)
except np.linalg.LinAlgError:
    cov_tw = sigma2_tw * np.linalg.pinv(X_twfe.T @ X_twfe)

# Treatment coefficient is the last one
twfe_coef = beta_twfe[-1]
twfe_se = np.sqrt(cov_tw[-1, -1])
twfe_t = twfe_coef / twfe_se
twfe_p = 2 * stats.t.sf(abs(twfe_t), n_tw - k_tw)

print("=== TWFE Estimate ===")
print(f"  Coefficient on D_it (treated): {twfe_coef:.4f}")
print(f"  Standard Error: {twfe_se:.4f}")
print(f"  t-statistic: {twfe_t:.3f}")
print(f"  p-value: {twfe_p:.6f}")
print(f"  95% CI: [{twfe_coef - 1.96*twfe_se:.4f}, {twfe_coef + 1.96*twfe_se:.4f}]")

# Compare to the true average treatment effect
true_att = df.loc[df['treated'] == 1, 'true_te'].mean()
print(f"\n  True average ATT: {true_att:.4f}")
print(f"  TWFE bias: {twfe_coef - true_att:.4f}")

# Visualize the staggered adoption
fig, ax = plt.subplots(figsize=(11, 6))
treated_cities = df[df['cohort'].notna()]['city'].unique()
never_treated = df[df['cohort'].isna()]['city'].unique()

colors_treated = ['#EE1515', '#FF7043', '#E040FB', '#FF4081']
for i, city in enumerate(sorted(treated_cities)):
    c = df[df['city'] == city]
    ax.plot(c['period'], c['avg_pokemon_level'], 'o-',
            color=colors_treated[i], linewidth=2, label=city, markersize=5)
    cohort_val = c['cohort'].iloc[0]
    ax.axvline(cohort_val, color=colors_treated[i], linestyle=':', alpha=0.4)

for city in sorted(never_treated):
    c = df[df['city'] == city]
    ax.plot(c['period'], c['avg_pokemon_level'], '--',
            color='#888888', linewidth=1, alpha=0.6, label=city)

ax.set_xlabel('Period', fontsize=13)
ax.set_ylabel('Avg Pokemon Level', fontsize=13)
ax.set_title('Staggered Shadow Surge Adoption', fontsize=14)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

blue_says(
    f"TWFE gives us {twfe_coef:.2f} as the treatment effect. The true ATT is {true_att:.2f}. "
    "Wait... those don't match! Something is wrong with TWFE in this staggered setting..."
)

## Cell 6: The TWFE Trap -- Goodman-Bacon Decomposition

This is the **big reveal**. Goodman-Bacon (2021) showed that the TWFE estimator is a weighted average of all possible 2x2 DiD comparisons. In a staggered setting, some of these comparisons use **already-treated** units as controls. If the treatment effect changes over time, these comparisons can produce **negative weights** and biased estimates.

In [ ]:
# ============================================================
# Cell 6: The TWFE Trap -- Goodman-Bacon Decomposition
# ============================================================

oak_says(
    "Goodman-Bacon (2021) showed that the TWFE estimator is a <b>weighted average</b> "
    "of all possible 2x2 DiD comparisons. With staggered treatment, some comparisons "
    "use already-treated units as 'controls.' If treatment effects vary over time, this "
    "produces <b>negative weights</b> -- and biased estimates. Let's decompose!"
)

# Implement a simplified Goodman-Bacon decomposition
# For each pair of treated cohorts (early vs late) and each pair of (treated vs never-treated)
# compute the 2x2 DiD and the weight

df = shadow.copy()
all_cities = df['city'].unique()
treated_cohorts = df[df['cohort'].notna()].groupby('city')['cohort'].first().to_dict()
never_treated_cities = df[df['cohort'].isna()]['city'].unique().tolist()

# Function to compute 2x2 DiD for any pair of groups and a treatment time
def did_2x2(df, treat_city, ctrl_city, treat_time, outcome='avg_pokemon_level'):
    """Compute 2x2 DiD for a specific pair."""
    t_data = df[df['city'] == treat_city]
    c_data = df[df['city'] == ctrl_city]
    y_pre_t = t_data[t_data['period'] < treat_time][outcome].mean()
    y_post_t = t_data[t_data['period'] >= treat_time][outcome].mean()
    y_pre_c = c_data[c_data['period'] < treat_time][outcome].mean()
    y_post_c = c_data[c_data['period'] >= treat_time][outcome].mean()
    return (y_post_t - y_pre_t) - (y_post_c - y_pre_c)

decomp_results = []
T = df['period'].max()

# Type 1: Treated vs Never-Treated
for treat_city, g in treated_cohorts.items():
    for ctrl_city in never_treated_cities:
        did_val = did_2x2(df, treat_city, ctrl_city, g)
        # Weight proportional to group sizes and variance of treatment
        n_treat_periods = T - g + 1
        n_ctrl_periods = g - 1
        wt = (n_treat_periods * n_ctrl_periods) / T**2
        decomp_results.append({
            'type': 'Treated vs Never-Treated',
            'treat': treat_city,
            'control': ctrl_city,
            'treat_time': g,
            'did': did_val,
            'weight': wt,
        })

# Type 2: Early vs Late (early treated, late as control)
for early_city, g_early in treated_cohorts.items():
    for late_city, g_late in treated_cohorts.items():
        if g_early >= g_late:
            continue
        # Early treated vs late (using early's treatment time)
        did_val = did_2x2(df, early_city, late_city, g_early)
        n_mid = g_late - g_early
        n_pre = g_early - 1
        wt = (n_mid * n_pre) / T**2
        decomp_results.append({
            'type': 'Early vs Late (clean)',
            'treat': early_city,
            'control': late_city,
            'treat_time': g_early,
            'did': did_val,
            'weight': wt,
        })

# Type 3: Late vs Early (late treated, early ALREADY-treated as control) -- THE PROBLEM
for late_city, g_late in treated_cohorts.items():
    for early_city, g_early in treated_cohorts.items():
        if g_late <= g_early:
            continue
        did_val = did_2x2(df, late_city, early_city, g_late)
        n_post_early = T - g_late + 1
        n_mid = g_late - g_early
        wt = (n_post_early * n_mid) / T**2
        decomp_results.append({
            'type': 'Late vs Already-Treated',
            'treat': late_city,
            'control': f"{early_city} (ALREADY TREATED)",
            'treat_time': g_late,
            'did': did_val,
            'weight': wt,
        })

decomp_df = pd.DataFrame(decomp_results)

# Normalize weights to sum to 1
decomp_df['weight_norm'] = decomp_df['weight'] / decomp_df['weight'].sum()
decomp_df['weighted_did'] = decomp_df['did'] * decomp_df['weight_norm']

print("=== Goodman-Bacon Decomposition ===")
print(f"Number of 2x2 comparisons: {len(decomp_df)}")
print()
for comp_type in decomp_df['type'].unique():
    sub = decomp_df[decomp_df['type'] == comp_type]
    print(f"--- {comp_type} ---")
    print(f"  Count: {len(sub)}")
    print(f"  Avg DiD: {sub['did'].mean():.3f}")
    print(f"  Total weight: {sub['weight_norm'].sum():.3f}")
    print(f"  Weighted contribution: {sub['weighted_did'].sum():.3f}")
    print()

bacon_total = decomp_df['weighted_did'].sum()
print(f"Bacon-weighted total: {bacon_total:.4f}")
print(f"TWFE coefficient:     {twfe_coef:.4f}")
print(f"True ATT:             {true_att:.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Scatter of weights vs DiD estimates
ax = axes[0]
type_colors = {
    'Treated vs Never-Treated': '#4DAD5B',
    'Early vs Late (clean)': '#3B4CCA',
    'Late vs Already-Treated': '#EE1515',
}
for comp_type, color in type_colors.items():
    sub = decomp_df[decomp_df['type'] == comp_type]
    ax.scatter(sub['weight_norm'], sub['did'], color=color, s=100,
              edgecolors='white', linewidth=0.8, label=comp_type, zorder=3)

ax.axhline(true_att, color='gold', linestyle='--', linewidth=2, label=f'True ATT = {true_att:.2f}')
ax.axhline(twfe_coef, color='gray', linestyle=':', linewidth=1.5, label=f'TWFE = {twfe_coef:.2f}')
ax.set_xlabel('Weight in TWFE', fontsize=12)
ax.set_ylabel('2x2 DiD Estimate', fontsize=12)
ax.set_title('Bacon Decomposition: Weights vs Estimates', fontsize=13)
ax.legend(fontsize=8)

# Panel 2: Stacked bar of contributions
ax = axes[1]
type_sums = decomp_df.groupby('type')['weighted_did'].sum()
bars = ax.bar(range(len(type_sums)), type_sums.values,
              color=[type_colors.get(t, '#888') for t in type_sums.index],
              edgecolor='white', width=0.6)
ax.set_xticks(range(len(type_sums)))
ax.set_xticklabels([t.replace(' ', '\n') for t in type_sums.index], fontsize=9)
ax.axhline(0, color='gray', linewidth=0.8)
ax.axhline(true_att, color='gold', linestyle='--', linewidth=2, label=f'True ATT')
for i, (bar, val) in enumerate(zip(bars, type_sums.values)):
    ax.text(bar.get_x() + bar.get_width()/2, val, f'{val:.2f}',
            ha='center', va='bottom' if val > 0 else 'top', fontweight='bold', fontsize=10)
ax.set_ylabel('Weighted Contribution', fontsize=12)
ax.set_title('Contributions to TWFE Estimate', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

blues_mistake(
    "TWFE with staggered treatment is fine -- it just runs a regression with unit and time fixed effects!",
    "TWFE uses already-treated units as controls, producing biased estimates when treatment effects "
    "are heterogeneous over time. The Bacon decomposition reveals that some 2x2 comparisons have "
    "problematic 'forbidden' comparisons that can bias the overall estimate."
)

## Cell 7: Callaway & Sant'Anna Estimator

Callaway and Sant'Anna (2021) proposed estimating **group-time ATTs** -- treatment effects specific to each cohort and time period. These are then aggregated properly, avoiding the forbidden comparisons that plague TWFE.

In [ ]:
# ============================================================
# Cell 7: Callaway-Sant'Anna Group-Time ATTs
# ============================================================

# Manual implementation of the core CS idea:
# For each cohort g and time t >= g, estimate ATT(g,t) using only
# not-yet-treated or never-treated units as controls

df = shadow.copy()
cohorts = sorted(df['cohort'].dropna().unique())
all_periods = sorted(df['period'].unique())
never_treated_mask = df['cohort'].isna()

def cs_att(df, cohort_g, time_t, outcome='avg_pokemon_level'):
    """
    Estimate ATT(g, t) using never-treated and not-yet-treated as controls.
    Uses a simple DiD: compare change from g-1 to t for cohort g vs control group.
    """
    # Treated group: cities in cohort g
    treated_cities = df[df['cohort'] == cohort_g]['city'].unique()
    
    # Control group: never-treated + not-yet-treated at time t
    control_mask = df['cohort'].isna() | (df['cohort'] > time_t)
    control_cities = df[control_mask]['city'].unique()
    
    if len(control_cities) == 0:
        return np.nan, np.nan
    
    base_period = cohort_g - 1  # one period before treatment
    
    # Treated: change from base to t
    y_t_base = df[(df['city'].isin(treated_cities)) & (df['period'] == base_period)][outcome].mean()
    y_t_post = df[(df['city'].isin(treated_cities)) & (df['period'] == time_t)][outcome].mean()
    
    # Control: change from base to t
    y_c_base = df[(df['city'].isin(control_cities)) & (df['period'] == base_period)][outcome].mean()
    y_c_post = df[(df['city'].isin(control_cities)) & (df['period'] == time_t)][outcome].mean()
    
    att = (y_t_post - y_t_base) - (y_c_post - y_c_base)
    return att, len(treated_cities)

# Compute group-time ATTs
gt_results = []
for g in cohorts:
    for t in all_periods:
        if t < g:  # pre-treatment: use for pre-trend check
            att, n = cs_att(df, g, t)
            gt_results.append({'cohort': int(g), 'time': t, 'rel_time': t - g,
                              'att': att, 'n_treated': n, 'post': False})
        elif t >= g:  # post-treatment
            att, n = cs_att(df, g, t)
            gt_results.append({'cohort': int(g), 'time': t, 'rel_time': t - g,
                              'att': att, 'n_treated': n, 'post': True})

gt_df = pd.DataFrame(gt_results)

# Aggregate: simple ATT (average over all post-treatment group-time cells)
post_gt = gt_df[gt_df['post'] == True]
cs_att_simple = post_gt['att'].mean()

# Event-study aggregation: average by rel_time across cohorts
es_agg = gt_df.groupby('rel_time')['att'].agg(['mean', 'std', 'count']).reset_index()
es_agg['se'] = es_agg['std'] / np.sqrt(es_agg['count'])
es_agg.loc[es_agg['se'].isna(), 'se'] = 0

print("=== Callaway-Sant'Anna Results ===")
print(f"  Simple ATT (avg of group-time ATTs): {cs_att_simple:.4f}")
print(f"  TWFE estimate:                        {twfe_coef:.4f}")
print(f"  True ATT:                             {true_att:.4f}")
print(f"  CS correction (vs TWFE):              {cs_att_simple - twfe_coef:+.4f}")
print()

# Group-time ATTs table
print("=== Group-Time ATTs (post-treatment only) ===")
pivot = post_gt.pivot_table(index='cohort', columns='time', values='att', aggfunc='mean')
print(pivot.round(2).to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Event study aggregation from CS
ax = axes[0]
# Only plot reasonable range
plot_range = es_agg[(es_agg['rel_time'] >= -5) & (es_agg['rel_time'] <= 12)]
ax.errorbar(plot_range['rel_time'], plot_range['mean'],
            yerr=1.96 * plot_range['se'],
            fmt='o-', color='#4DAD5B', capsize=4, linewidth=2, markersize=7,
            label='CS Event Study')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(-0.5, color='#FFD733', linestyle='--', linewidth=2, label='Treatment onset')
ax.set_xlabel('Periods Relative to Treatment', fontsize=12)
ax.set_ylabel('Average ATT', fontsize=12)
ax.set_title('Callaway-Sant\'Anna Event Study', fontsize=13)
ax.legend(fontsize=10)

# Panel 2: Comparison bar chart
ax = axes[1]
estimators = ['TWFE', 'Callaway-Sant\'Anna', 'True ATT']
values = [twfe_coef, cs_att_simple, true_att]
colors_bar = ['#EE1515', '#4DAD5B', '#FFD733']
bars = ax.bar(estimators, values, color=colors_bar, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Estimated ATT', fontsize=12)
ax.set_title('TWFE vs CS vs Truth', fontsize=13)
ax.axhline(0, color='gray', linewidth=0.5)

plt.tight_layout()
plt.show()

oak_says(
    f"The Callaway-Sant'Anna estimator gives an ATT of <b>{cs_att_simple:.2f}</b>, "
    f"much closer to the true ATT of <b>{true_att:.2f}</b> than the TWFE estimate of "
    f"<b>{twfe_coef:.2f}</b>. By only using clean comparisons (never-treated and not-yet-treated "
    "as controls), we avoid the bias from forbidden comparisons."
)

## Cell 8: Synthetic Control -- Building Synthetic Saffron

Now we switch methods entirely. Instead of finding one control city, we **build** a synthetic version of Saffron City from a weighted combination of control cities. This is the **Synthetic Control Method** (Abadie, Diamond & Hainmueller, 2010).

Setting: Team Rocket invaded Saffron City in period 8. We want to estimate what Saffron's trainer level would have been without the invasion.

In [ ]:
# ============================================================
# Cell 8: Synthetic Control Method
# ============================================================

# Use kanto_cities_panel for the Team Rocket invasion setting
panel = cities.copy()
treatment_period_sc = 8  # Team Rocket invaded Saffron in period 8
treated_unit = 'Saffron City'

# Donor pool: cities that were NOT invaded by Team Rocket
# Celadon was also invaded, so exclude it
donor_cities = [c for c in panel['city'].unique()
                if c != treated_unit and panel[panel['city'] == c]['team_rocket_invasion'].max() == 0]

print(f"Treated unit: {treated_unit}")
print(f"Treatment period: {treatment_period_sc}")
print(f"Donor cities: {donor_cities}")
print()

# Pivot to wide format: rows = periods, columns = cities
outcome_col = 'avg_trainer_level'
wide = panel.pivot_table(index='period', columns='city', values=outcome_col)

# Pre-treatment period
pre = wide.loc[wide.index < treatment_period_sc]
post = wide.loc[wide.index >= treatment_period_sc]

# Treated unit vector (pre-treatment)
y_treat_pre = pre[treated_unit].values
# Donor matrix (pre-treatment)
X_donors_pre = pre[donor_cities].values

# Optimization: find weights w >= 0, sum(w) = 1 that minimize
# || y_treat_pre - X_donors_pre @ w ||^2
n_donors = len(donor_cities)

def sc_objective(w):
    """Mean squared prediction error for synthetic control."""
    synthetic = X_donors_pre @ w
    return np.sum((y_treat_pre - synthetic)**2)

# Constraints: weights sum to 1
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
# Bounds: each weight between 0 and 1
bounds = [(0, 1)] * n_donors
# Initial weights: uniform
w0 = np.ones(n_donors) / n_donors

result_sc = optimize.minimize(sc_objective, w0, method='SLSQP',
                              bounds=bounds, constraints=constraints,
                              options={'maxiter': 1000, 'ftol': 1e-12})
w_star = result_sc.x

# Display weights
print("=== Synthetic Control Weights ===")
for donor, weight in sorted(zip(donor_cities, w_star), key=lambda x: -x[1]):
    if weight > 0.001:
        print(f"  {donor}: {weight:.4f}")
print()

# Construct synthetic Saffron (full time series)
all_periods_wide = wide[donor_cities].values
synthetic_saffron = all_periods_wide @ w_star
actual_saffron = wide[treated_unit].values
periods_arr = wide.index.values

# Compute the gap (actual - synthetic)
gap = actual_saffron - synthetic_saffron

# Pre-treatment fit
pre_rmse = np.sqrt(np.mean(gap[periods_arr < treatment_period_sc]**2))
post_avg_gap = np.mean(gap[periods_arr >= treatment_period_sc])

print(f"Pre-treatment RMSE: {pre_rmse:.3f}")
print(f"Average post-treatment gap (treatment effect): {post_avg_gap:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Actual vs Synthetic
ax = axes[0]
ax.plot(periods_arr, actual_saffron, 'o-', color='#EE1515', linewidth=2.5,
        markersize=6, label=f'Actual {treated_unit}')
ax.plot(periods_arr, synthetic_saffron, 's--', color='#3B4CCA', linewidth=2,
        markersize=5, label=f'Synthetic {treated_unit}')
ax.axvline(treatment_period_sc, color='#FFD733', linestyle='--', linewidth=2,
           label='Team Rocket Invasion')
ax.fill_between(periods_arr[periods_arr >= treatment_period_sc],
                actual_saffron[periods_arr >= treatment_period_sc],
                synthetic_saffron[periods_arr >= treatment_period_sc],
                color='#EE1515', alpha=0.15, label='Treatment Effect')
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel(outcome_col, fontsize=12)
ax.set_title('Synthetic Control: Saffron City', fontsize=13)
ax.legend(fontsize=9)

# Panel 2: Gap plot
ax = axes[1]
ax.plot(periods_arr, gap, 'o-', color='#EE1515', linewidth=2, markersize=6)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(treatment_period_sc, color='#FFD733', linestyle='--', linewidth=2,
           label='Treatment onset')
ax.fill_between(periods_arr[periods_arr >= treatment_period_sc],
                gap[periods_arr >= treatment_period_sc], 0,
                color='#EE1515', alpha=0.2)
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel('Gap (Actual - Synthetic)', fontsize=12)
ax.set_title('Gap Plot: Treatment Effect Over Time', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

oak_says(
    f"The synthetic control closely matches Saffron City's pre-treatment trend "
    f"(RMSE = {pre_rmse:.2f}). After the Team Rocket invasion, the gap between actual "
    f"and synthetic Saffron averages <b>{post_avg_gap:.2f}</b> trainer levels. But is this "
    "gap statistically meaningful? We need placebo tests to find out!"
)

## Cell 9: Synthetic Control Placebo Tests

To assess significance, we apply the synthetic control method to **every donor city** as if it were treated. If Saffron's gap is extreme relative to placebo gaps, we have evidence of a real effect.

In [ ]:
# ============================================================
# Cell 9: Placebo Tests for Synthetic Control
# ============================================================

def synthetic_control(panel_wide, treated, donors, treatment_period):
    """Run synthetic control and return gap series."""
    pre = panel_wide.loc[panel_wide.index < treatment_period]
    y_pre = pre[treated].values
    X_pre = pre[donors].values
    n_d = len(donors)
    
    def obj(w):
        return np.sum((y_pre - X_pre @ w)**2)
    
    cons = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
    bnds = [(0, 1)] * n_d
    w0 = np.ones(n_d) / n_d
    res = optimize.minimize(obj, w0, method='SLSQP', bounds=bnds,
                           constraints=cons, options={'maxiter': 1000, 'ftol': 1e-12})
    
    synthetic = panel_wide[donors].values @ res.x
    actual = panel_wide[treated].values
    gap = actual - synthetic
    pre_rmse = np.sqrt(np.mean(gap[panel_wide.index < treatment_period]**2))
    return gap, pre_rmse

# Run placebo tests for each donor city
placebo_gaps = {}
placebo_rmses = {}

# Saffron's gap (already computed)
placebo_gaps[treated_unit] = gap
placebo_rmses[treated_unit] = pre_rmse

for placebo_city in donor_cities:
    # Donors for this placebo: all other donor cities (not the placebo city itself)
    placebo_donors = [c for c in donor_cities if c != placebo_city]
    try:
        g, r = synthetic_control(wide, placebo_city, placebo_donors, treatment_period_sc)
        placebo_gaps[placebo_city] = g
        placebo_rmses[placebo_city] = r
    except Exception:
        pass

# Filter out placebo cities with poor pre-treatment fit
# (standard practice: exclude if pre-RMSE > X times Saffron's)
rmse_threshold = 5 * pre_rmse
good_placebos = {c: g for c, g in placebo_gaps.items()
                 if placebo_rmses[c] <= rmse_threshold}

print(f"Placebo cities with acceptable fit: {len(good_placebos) - 1} "
      f"(out of {len(donor_cities)})")
print()

# Plot all placebo gaps
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Spaghetti plot of all gaps
ax = axes[0]
for city_name, gap_series in good_placebos.items():
    if city_name == treated_unit:
        continue
    ax.plot(periods_arr, gap_series, color='#BBBBBB', linewidth=0.8, alpha=0.6)

# Saffron on top
ax.plot(periods_arr, placebo_gaps[treated_unit], 'o-', color='#EE1515',
        linewidth=2.5, markersize=5, label=treated_unit, zorder=5)
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(treatment_period_sc, color='#FFD733', linestyle='--', linewidth=2)
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel('Gap (Actual - Synthetic)', fontsize=12)
ax.set_title('Placebo Test: All Gaps', fontsize=13)
ax.legend(fontsize=10)

# Panel 2: Post/pre MSPE ratio (exact p-value)
ax = axes[1]
ratios = {}
for city_name, gap_series in good_placebos.items():
    post_mspe = np.mean(gap_series[periods_arr >= treatment_period_sc]**2)
    pre_mspe = np.mean(gap_series[periods_arr < treatment_period_sc]**2)
    if pre_mspe > 0:
        ratios[city_name] = post_mspe / pre_mspe

# Sort and plot
sorted_ratios = sorted(ratios.items(), key=lambda x: x[1], reverse=True)
city_names_sorted = [c for c, _ in sorted_ratios]
ratio_values = [r for _, r in sorted_ratios]

colors_bar = ['#EE1515' if c == treated_unit else '#3B4CCA' for c in city_names_sorted]
bars = ax.barh(range(len(city_names_sorted)), ratio_values, color=colors_bar,
               edgecolor='white', height=0.6)
ax.set_yticks(range(len(city_names_sorted)))
ax.set_yticklabels(city_names_sorted, fontsize=10)
ax.set_xlabel('Post/Pre MSPE Ratio', fontsize=12)
ax.set_title('Placebo Ranking (Higher = More Extreme)', fontsize=13)
ax.invert_yaxis()

# Compute p-value
saffron_rank = city_names_sorted.index(treated_unit) + 1
sc_p_value = saffron_rank / len(city_names_sorted)

plt.tight_layout()
plt.show()

print(f"\n=== Inference ===")
print(f"Saffron City rank: {saffron_rank} out of {len(city_names_sorted)}")
print(f"Exact p-value: {sc_p_value:.3f}")

oak_says(
    f"Saffron City's gap is ranked <b>#{saffron_rank}</b> out of {len(city_names_sorted)} cities "
    f"in terms of the post/pre MSPE ratio, giving an exact p-value of <b>{sc_p_value:.3f}</b>. "
    "The placebo test confirms that Saffron's post-invasion gap is unusually large -- "
    "much larger than what we'd expect from random variation alone."
)

---

## Challenges

Time to test your skills, trainer! Complete these challenges to earn the Marsh Badge.

### Challenge 1: DiD with a Different Treatment/Control Pair

Run a 2x2 DiD using **Cerulean City** (treated, Shadow Surge adopted in period 10) and **Fuchsia City** (never treated) from the staggered dataset. Compare your estimate to the true treatment effect for Cerulean.

In [ ]:
# ============================================================
# Challenge 1: DiD with Cerulean vs Fuchsia
# ============================================================

# YOUR CODE HERE
# 1. Filter the shadow surge data for Cerulean City and Fuchsia City
# 2. Set treatment period = 10 (Cerulean's adoption time)
# 3. Compute the 2x2 DiD by hand
# 4. Use did_estimate() to get standard errors
# 5. Compare to the true_te column for Cerulean post-treatment
# 6. Visualize with did_plot()

challenge_df = shadow[shadow['city'].isin(['Cerulean City', 'Fuchsia City'])].copy()
challenge_df['treated_grp'] = (challenge_df['city'] == 'Cerulean City').astype(int)
cerulean_treat_period = 10

# Extract pre/post values for both groups
cerulean = challenge_df[challenge_df['city'] == 'Cerulean City']
fuchsia = challenge_df[challenge_df['city'] == 'Fuchsia City']

y_pre_t = cerulean[cerulean['period'] < cerulean_treat_period]['avg_pokemon_level'].values
y_post_t = cerulean[cerulean['period'] >= cerulean_treat_period]['avg_pokemon_level'].values
y_pre_c = fuchsia[fuchsia['period'] < cerulean_treat_period]['avg_pokemon_level'].values
y_post_c = fuchsia[fuchsia['period'] >= cerulean_treat_period]['avg_pokemon_level'].values

result_ch1 = did_estimate(y_pre_t, y_post_t, y_pre_c, y_post_c)
true_te_cerulean = cerulean[cerulean['treated'] == 1]['true_te'].mean()

print("=== Challenge 1: Cerulean vs Fuchsia ===")
print(f"DiD Estimate: {result_ch1['estimate']:.4f}")
print(f"95% CI: [{result_ch1['ci_lower']:.4f}, {result_ch1['ci_upper']:.4f}]")
print(f"True ATT for Cerulean: {true_te_cerulean:.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
did_plot(challenge_df, time_col='period', outcome_col='avg_pokemon_level',
         group_col='treated_grp', treatment_period=cerulean_treat_period, ax=ax)
ax.set_title('Challenge 1: Cerulean City vs Fuchsia City', fontsize=14)
plt.tight_layout()
plt.show()

### Challenge 2: Sun-Abraham Estimator

Implement the **Sun & Abraham (2021)** interaction-weighted estimator. The key idea: instead of a single treatment dummy, interact cohort indicators with relative time indicators, then aggregate properly.

In [ ]:
# ============================================================
# Challenge 2: Sun-Abraham Estimator (Manual Implementation)
# ============================================================

# YOUR CODE HERE
# The Sun-Abraham estimator:
# 1. Run a fully-interacted regression: Y = unit_FE + time_FE + sum_{g,l} delta_{g,l} * 1[G=g] * 1[rel_time=l]
# 2. Aggregate: ATT_l = sum_g (share_g * delta_{g,l})
#    where share_g = (n_g / n_treated) is the share of treated units in cohort g

df_sa = shadow.copy()
cohorts_sa = sorted(df_sa['cohort'].dropna().unique())
periods_sa = sorted(df_sa['period'].unique())

# Step 1: For each cohort g and relative time l, estimate CATT(g,l)
# using only never-treated as controls (cleaner than TWFE)
sa_results = []
for g in cohorts_sa:
    cohort_cities = df_sa[df_sa['cohort'] == g]['city'].unique()
    control_cities_sa = df_sa[df_sa['cohort'].isna()]['city'].unique()
    
    for l in range(-5, 13):  # relative time window
        t = g + l  # actual period
        if t < 1 or t > 24:
            continue
        base = g - 1  # reference period
        if base < 1:
            continue
        
        # DiD: (Y_treated_t - Y_treated_base) - (Y_control_t - Y_control_base)
        y_t_t = df_sa[(df_sa['city'].isin(cohort_cities)) & (df_sa['period'] == t)]['avg_pokemon_level'].mean()
        y_t_b = df_sa[(df_sa['city'].isin(cohort_cities)) & (df_sa['period'] == base)]['avg_pokemon_level'].mean()
        y_c_t = df_sa[(df_sa['city'].isin(control_cities_sa)) & (df_sa['period'] == t)]['avg_pokemon_level'].mean()
        y_c_b = df_sa[(df_sa['city'].isin(control_cities_sa)) & (df_sa['period'] == base)]['avg_pokemon_level'].mean()
        
        catt = (y_t_t - y_t_b) - (y_c_t - y_c_b)
        n_g = len(cohort_cities)
        sa_results.append({'cohort': int(g), 'rel_time': l, 'catt': catt, 'n_g': n_g})

sa_df = pd.DataFrame(sa_results)

# Step 2: Aggregate across cohorts using cohort shares
total_treated = sum(len(df_sa[df_sa['cohort'] == g]['city'].unique()) for g in cohorts_sa)
sa_df['share'] = sa_df['n_g'] / total_treated

sa_agg = sa_df.groupby('rel_time').apply(
    lambda x: np.average(x['catt'], weights=x['share'])
).reset_index(name='sa_att')

# Compare CS and SA event studies
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(sa_agg['rel_time'], sa_agg['sa_att'], 'D-', color='#7B61FF',
        linewidth=2, markersize=7, label='Sun-Abraham')

# Overlay CS for comparison
cs_plot = es_agg[(es_agg['rel_time'] >= -5) & (es_agg['rel_time'] <= 12)]
ax.plot(cs_plot['rel_time'], cs_plot['mean'], 's--', color='#4DAD5B',
        linewidth=1.5, markersize=5, alpha=0.7, label='Callaway-Sant\'Anna')

ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(-0.5, color='#FFD733', linestyle='--', linewidth=2, label='Treatment onset')
ax.set_xlabel('Periods Relative to Treatment', fontsize=12)
ax.set_ylabel('Estimated ATT', fontsize=12)
ax.set_title('Challenge 2: Sun-Abraham vs Callaway-Sant\'Anna', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

sa_overall = sa_df[sa_df['rel_time'] >= 0]['catt'].mean()
print(f"Sun-Abraham overall ATT: {sa_overall:.4f}")
print(f"CS overall ATT:          {cs_att_simple:.4f}")
print(f"TWFE:                    {twfe_coef:.4f}")
print(f"True ATT:                {true_att:.4f}")

### Challenge 3: Different SC Donor Pools

Try different synthetic control donor pools and compare how the weights and estimated treatment effects change.

In [ ]:
# ============================================================
# Challenge 3: Different Donor Pools for Synthetic Control
# ============================================================

# YOUR CODE HERE
# Compare three donor pools:
# Pool 1: All non-invaded cities (original)
# Pool 2: Only larger cities (Cerulean, Vermilion, Fuchsia)
# Pool 3: All cities except Saffron (including Celadon, which was also invaded)

pools = {
    'Original (no invasion)': [c for c in panel['city'].unique()
                               if c != treated_unit and panel[panel['city'] == c]['team_rocket_invasion'].max() == 0],
    'Larger cities only': ['Cerulean City', 'Vermilion City', 'Fuchsia City'],
    'All except Saffron': [c for c in panel['city'].unique() if c != treated_unit],
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pool_results = {}

for idx, (pool_name, donors) in enumerate(pools.items()):
    ax = axes[idx]
    gap_pool, rmse_pool = synthetic_control(wide, treated_unit, donors, treatment_period_sc)
    
    actual = wide[treated_unit].values
    synthetic_vals = actual - gap_pool
    post_gap = np.mean(gap_pool[periods_arr >= treatment_period_sc])
    
    # Get weights
    pre_p = wide.loc[wide.index < treatment_period_sc]
    y_p = pre_p[treated_unit].values
    X_p = pre_p[donors].values
    n_d = len(donors)
    r = optimize.minimize(lambda w: np.sum((y_p - X_p @ w)**2),
                         np.ones(n_d)/n_d, method='SLSQP',
                         bounds=[(0,1)]*n_d,
                         constraints={'type':'eq','fun':lambda w: np.sum(w)-1})
    
    ax.plot(periods_arr, actual, 'o-', color='#EE1515', linewidth=2, markersize=4, label='Actual')
    ax.plot(periods_arr, synthetic_vals, 's--', color='#3B4CCA', linewidth=1.5, markersize=3, label='Synthetic')
    ax.axvline(treatment_period_sc, color='#FFD733', linestyle='--', linewidth=1.5)
    ax.set_title(f'{pool_name}\nGap={post_gap:.2f}, RMSE={rmse_pool:.2f}', fontsize=11)
    ax.set_xlabel('Period')
    if idx == 0:
        ax.set_ylabel('Avg Trainer Level')
    ax.legend(fontsize=8)
    
    # Print weights
    print(f"\n=== {pool_name} ===")
    print(f"  Post-treatment gap: {post_gap:.3f}")
    print(f"  Pre-RMSE: {rmse_pool:.3f}")
    print("  Weights:")
    for d, w in sorted(zip(donors, r.x), key=lambda x: -x[1]):
        if w > 0.001:
            print(f"    {d}: {w:.4f}")

plt.tight_layout()
plt.show()

### Challenge 4: Synthetic Difference-in-Differences

Combine synthetic control weights with DiD logic: use SC weights to construct a parallel control, then apply DiD. This is the **Synthetic DiD** idea from Arkhangelsky et al. (2021).

In [ ]:
# ============================================================
# Challenge 4: Synthetic Difference-in-Differences
# ============================================================

# YOUR CODE HERE
# Idea: SDID combines unit weights (like SC) with time weights (like DiD)
# Simplified version:
# 1. Compute SC unit weights (already have w_star)
# 2. Compute time weights: emphasize pre-treatment periods that best predict
#    the treatment period (optional simplification: use uniform time weights)
# 3. Estimate ATT as: (Y_treated_post - Y_treated_pre) - (Y_synth_post - Y_synth_pre)

# Step 1: SC weights (from Cell 8)
print("SC unit weights:")
for d, w in zip(donor_cities, w_star):
    if w > 0.001:
        print(f"  {d}: {w:.4f}")

# Step 2: Construct synthetic control series using SC weights
Y_synth = wide[donor_cities].values @ w_star
Y_actual = wide[treated_unit].values

# Step 3: Apply DiD on top
pre_mask_sdid = periods_arr < treatment_period_sc
post_mask_sdid = periods_arr >= treatment_period_sc

# Pure SC estimate
sc_est = np.mean(Y_actual[post_mask_sdid] - Y_synth[post_mask_sdid])

# SDID estimate: remove the pre-treatment level difference
pre_gap = np.mean(Y_actual[pre_mask_sdid] - Y_synth[pre_mask_sdid])
sdid_est = np.mean(Y_actual[post_mask_sdid] - Y_synth[post_mask_sdid]) - pre_gap

# Also compute time-weighted version
# Time weights: penalize early periods, emphasize periods close to treatment
pre_periods_arr = periods_arr[pre_mask_sdid]
time_weights = np.exp(-0.5 * (treatment_period_sc - pre_periods_arr))
time_weights = time_weights / time_weights.sum()

pre_gap_weighted = np.sum(time_weights * (Y_actual[pre_mask_sdid] - Y_synth[pre_mask_sdid]))
sdid_weighted = np.mean(Y_actual[post_mask_sdid] - Y_synth[post_mask_sdid]) - pre_gap_weighted

print(f"\n=== Synthetic DiD Results ===")
print(f"Pure SC estimate:                  {sc_est:.4f}")
print(f"SDID (uniform time weights):       {sdid_est:.4f}")
print(f"SDID (exponential time weights):   {sdid_weighted:.4f}")
print(f"Classic DiD (Saffron vs Pewter):    {did_manual:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(periods_arr, Y_actual, 'o-', color='#EE1515', linewidth=2.5,
        markersize=6, label='Actual Saffron')
ax.plot(periods_arr, Y_synth, 's--', color='#3B4CCA', linewidth=2,
        markersize=5, label='SC Synthetic')

# SDID counterfactual: shift synthetic up by pre-treatment gap
Y_sdid_cf = Y_synth + pre_gap
ax.plot(periods_arr, Y_sdid_cf, 'D:', color='#4DAD5B', linewidth=2,
        markersize=4, label='SDID Counterfactual')

ax.axvline(treatment_period_sc, color='#FFD733', linestyle='--', linewidth=2,
           label='Treatment')
ax.fill_between(periods_arr[post_mask_sdid],
                Y_actual[post_mask_sdid],
                Y_sdid_cf[post_mask_sdid],
                color='#4DAD5B', alpha=0.15, label=f'SDID effect = {sdid_est:.2f}')
ax.set_xlabel('Period', fontsize=12)
ax.set_ylabel('Avg Trainer Level', fontsize=12)
ax.set_title('Synthetic Difference-in-Differences', fontsize=14)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

oak_says(
    "Synthetic DiD combines the best of both worlds: SC's data-driven weighting "
    "of control units with DiD's differencing to remove level differences. The SDID "
    "counterfactual accounts for any remaining pre-treatment gap between actual and "
    "synthetic Saffron."
)

---

## Badge Earned!

In [ ]:
# ============================================================
# Marsh Badge
# ============================================================

badge_earned("Marsh Badge", 7)

oak_says(
    "Outstanding work, trainer! You've mastered the arts of Saffron City: "
    "classic DiD, parallel trends testing, event studies, TWFE and its pitfalls "
    "with staggered treatment, the Goodman-Bacon decomposition, Callaway-Sant'Anna "
    "estimation, and synthetic control methods. The Marsh Badge is yours! "
    "Next stop: Indigo Plateau, where the Elite Four await with advanced topics."
)